# R19-H195 - the hard-probe instrument and the retrieval-convention audit

**Clause (a)** re-scores the three-arm benchmark (arms 1-2) and the H53 top_k 8-vs-16 comparison under the
generous-top_k-truncation convention (query HNSW at top_k=64, truncate to k) AND the H194 router scorer (deterministic
value comparator + word-overlap). Reports per-arm old-vs-new magnitudes and whether any recorded VERDICT flips
(bar: no flip, shifts <= 8pt). Arm 3 (SOTA rebuild, default `neo4j`) was wiped for H158; its cache has no seed lists -> NOT RE-SCORABLE.

**Clause (b)** builds a difficulty-engineered wide benchmark on the H188 derivation harness (catalogue + spec-proximity +
deterministic multi-doc consolidation golds; feature names quarantined to a coverage tier). Target >= 200 golds,
document-grounded, recall@8 in the 45-70% band with a k-lever >= 8pt, on neo4j2 under the correct convention + router.
Saved to `data/processed/probes-wide-v2-h195.json`. `neo4j2` is READ-ONLY.

In [1]:
import os, warnings, datetime
warnings.filterwarnings("ignore")
os.environ["NEO4J_URI"]="bolt://user-konrad.jelen-kgf-neo4j2:7687"   # READ-ONLY 10-doc benchmark reference
os.environ["NEO4J_USER"]="neo4j"; os.environ["NEO4J_PASSWORD"]="kgfoundry"
import re, json, pickle, hashlib, unicodedata, itertools, collections
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, yaml
from rich import print as rprint
from knowledge_graph_foundry import load_settings, Foundry
from knowledge_graph_foundry.graph.graphrag import vector_query
from knowledge_graph_foundry.extraction import generate_embeddings
from knowledge_graph_foundry.models import Entity as KEnt
from neo4j import GraphDatabase
np.random.seed(42)
STAMP=datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
settings=load_settings(Path("../config.yml")); VEC_INDEX=settings.graphrag.vector_index_name; REL_LIMIT=15
rprint(f"[cyan]vec_index={VEC_INDEX} stamp={STAMP}[/cyan]")

2026-07-07 17:21:06.883 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


vec_index=kgf_entity_embeddings stamp=20260707T152107Z

In [2]:
# --- graph pull (READ-ONLY neo4j2) + render primitives + H194 router (comparator + word-overlap) ---
d=GraphDatabase.driver(os.environ["NEO4J_URI"], auth=("neo4j","kgfoundry"))
with d.session() as s:
    ents=s.run("MATCH (e:Entity) RETURN e.id AS id,e.name AS name,e.description AS description,"
               "properties(e) AS props,labels(e) AS types").data()
    edges=s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                "RETURN DISTINCT a.id AS a,b.id AS b,type(r) AS rel").data()
    prop_rows=s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid,p.text AS text").data()
    alias_rows=s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                     "RETURN e.id AS eid,collect(DISTINCT a.id)[..5] AS aliases").data()
d.close()
node={r["id"]:r for r in ents}; names={r["id"]:r["name"] for r in ents}
props_by=defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by={r["eid"]:r["aliases"] for r in alias_rows}
rels_by=defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"],e["b"])); rels_by[e["b"]].append((e["rel"],e["a"]))
def spec_of(r): return {k.removeprefix("prop_"):v for k,v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r=node[nid]; spec=dict(spec_of(r))
    for a in [a for a in alias_by.get(nid,[]) if a in node]:
        for k,v in spec_of(node[a]).items(): spec.setdefault(k,v)
    return spec
def base_render(nid):
    r=node[nid]; spec=merged_spec(nid); al=[a for a in alias_by.get(nid,[]) if a in node]
    aka=(f"Also known as: {', '.join(names.get(a,'') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid): return base_render(nid)+" "+" ; ".join(f"{t} -> {names.get(b,'')}" for t,b in rels_by.get(nid,[])[:REL_LIMIT])
def units_of(ids): return [seed_render(n) for n in ids] + [t for n in ids for t in props_by.get(n,[])]

_TM=dict.fromkeys(map(ord,"®™©"),None)
def gnorm(s):
    s=(s or "").translate(_TM); s=unicodedata.normalize("NFKC",s)
    s=s.replace(" "," ").replace("×","x").replace("*","x").replace("·","x")
    s=re.sub(r"(?<=\d),(?=\d)","",s)
    return re.sub(r"\s+"," ",s.casefold()).strip()
UNITWORD={"mm":"len_mm","cm":"len_cm","g":"mass_g","kg":"mass_kg","oz":"mass_oz","ml":"vol_ml","l":"vol_l",
   "db":"sound_db","dba":"sound_db","w":"power_w","hz":"freq_hz","cmh2o":"press","m":"alt_m",
   "min":"time_min","mins":"time_min","minute":"time_min","minutes":"time_min","year":"warr_y","years":"warr_y"}
FAM_EQ={"len_mm":{"len_mm"},"len_cm":{"len_cm"},"alt_m":{"alt_m"},"mass_g":{"mass_g"},"mass_kg":{"mass_kg"},
   "mass_oz":{"mass_oz"},"vol_ml":{"vol_ml","vol_l"},"sound_db":{"sound_db"},"power_w":{"power_w"},
   "time_min":{"time_min"},"warr_y":{"warr_y"},"press":{"press"},"freq_hz":{"freq_hz"}}
def key_family(k):
    k=k.lower()
    if "dimension" in k or re.search(r"_mm\b",k) or "length_mm" in k: return "len_mm"
    if "altitude" in k: return "alt_m"
    if k.endswith("_kg") or "weight_kg" in k: return "mass_kg"
    if re.search(r"_g\b",k): return "mass_g"
    if "_oz" in k: return "mass_oz"
    if re.search(r"_ml\b",k) or "capacity_ml" in k or "water" in k: return "vol_ml"
    if "sound" in k or re.search(r"_db\b",k) or "noise" in k: return "sound_db"
    if "power" in k or "consumption" in k: return "power_w"
    if "ramp" in k or "delay" in k: return "time_min"
    if "warranty" in k: return "warr_y"
    if "pressure" in k: return "press"
    return None
def nums_in(v): return re.findall(r"\d+(?:\.\d+)?", str(v).replace(",",""))
def ctx_quantities(ids):
    Q=set()
    for nid in ids:
        spec=merged_spec(nid); unit_for={}
        for k,v in spec.items():
            if k.endswith("_unit"):
                fam=UNITWORD.get(gnorm(str(v)).replace(" ",""))
                if fam: unit_for[k[:-5]]=fam
        for k,v in spec.items():
            fam=key_family(k) or unit_for.get(k)
            if fam:
                for n in nums_in(v): Q.add((n,fam))
        text=gnorm(seed_render(nid)+" "+" ".join(props_by.get(nid,[])))
        for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", text):
            fam=UNITWORD.get(m.group(2).replace("(a)",""))
            if fam: Q.add((m.group(1),fam))
    return Q
def parse_gold(gold):
    g=gnorm(gold)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return ("dim", re.findall(r"\d+(?:\.\d+)?", g))
    if "sd card" in g: return ("sdcard", None)
    m=re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|cm h2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", g)
    rng=re.search(r"(\d+(?:\.\d+)?)\s*(?:to|-)\s*(\d+(?:\.\d+)?)", g)
    if m:
        u=m.group(2).replace("(a)","").replace("cm h2o","cmh2o").replace(" ",""); fam=UNITWORD.get(u)
        if rng and rng.group(2): return ("range",(rng.group(1),rng.group(2),fam))
        return ("num",(m.group(1),fam))
    if rng and rng.group(2):
        fam="press" if "cmh2o" in g or "cm h2o" in g else ("time_min" if "min" in g else None)
        return ("range",(rng.group(1),rng.group(2),fam))
    return ("other", None)
def comparator(gold, ids):
    kind,payload=parse_gold(gold); Q=ctx_quantities(ids)
    T=gnorm(" ".join(seed_render(n)+" "+" ".join(props_by.get(n,[])) for n in ids))
    if kind=="dim":
        a,b,c=payload; mm={n for n,f in Q if f=="len_mm"}
        if {a,b,c}<=mm: return True
        t=T.replace(" ","")
        return any(re.search(r"(?<!\d)"+p[0]+"x"+p[1]+"x"+p[2]+r"(?!\d)", t) for p in itertools.permutations([a,b,c]))
    if kind=="num":
        n,fam=payload
        if fam is None: return any(x==n for x,_ in Q)
        eq=FAM_EQ.get(fam,{fam}); return any(x==n and f in eq for x,f in Q)
    if kind=="range":
        a,b,fam=payload; t=T.replace(" ","")
        if re.search(r"(?<!\d)"+a+r"\s*-\s*"+b,T) or (a+"-"+b) in t or (a+"to"+b) in t: return True
        if fam:
            eq=FAM_EQ.get(fam,{fam}); xs={x for x,f in Q if f in eq}; return a in xs and b in xs
        return False
    if kind=="sdcard":
        t=T.replace(" ",""); return ("sdcard" in t) and (">1year" in t or "1year" in t)
    return None
def word_overlap(gold, ids, thr=0.6):
    ng=gnorm(gold); ctx=gnorm(" ".join(units_of(ids)))
    w=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ng)); cw=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ctx))
    return bool(w) and len(w&cw)/len(w)>=thr
def router_present(gold, ids):
    c=comparator(gold, ids)
    return bool(c) if c is not None else word_overlap(gold, ids)

# --- incumbent fuzzy matcher (three_arm/H53 "present") ---
def _norm(s): return re.sub(r"\s+"," ",(s or "").casefold())
def value_tokens(t): return re.findall(r"[\w.\-/]*\d[\w.\-/]*", t)
_UNIT=r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"
def fuzzy_present(gold, ctx):
    ng=_norm(gold)
    if ng in ctx: return True
    sq=re.sub(r"[\s,()]","",ctx); sk=re.sub(r"[\s,()]","",re.sub(_UNIT,"",ng))
    if any(c.isdigit() for c in sk) and len(sk)>=5 and sk in sq: return True
    tok=value_tokens(gold)
    if tok:
        hit=sum(1 for t in tok if _norm(t) in ctx or re.sub(r"[\s,()]","",_norm(t)) in sq)
        return hit>=max(1,len(tok)//2+(len(tok)%2))
    w=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ng)); cw=set(re.findall(r"[a-z][a-z0-9\-]{2,}",ctx))
    return bool(w) and len(w&cw)/len(w)>=0.6


# --- lean three_arm render_nodes (for the convention decomposition) ---
def lean_render(ids):
    blocks=[]
    for nid in ids:
        r=node.get(nid)
        if r is None: continue
        spec={k.removeprefix("prop_"):v for k,v in r["props"].items() if k.startswith("prop_")}
        rl=rels_by.get(nid,[])[:15]
        blocks.append(f"## {r['name']} ({', '.join(r['types'])})\n{r['description'] or ''}\n"
                      f"Properties: {json.dumps(spec, default=str)}\nRelations: "+
                      "; ".join(f"{t} -> {names.get(b,'')}" for t,b in rl))
    return _norm("\n".join(blocks))
rprint("[green]harness ready[/green] router + fuzzy + lean render")


harness ready router + fuzzy + lean render

## Clause (a) - retrieval-convention audit

In [3]:
# H195(a) - convention audit. Probes = 33-gold/24-probe CPAP set (same set the three-arm & H53 used).
probes=yaml.safe_load(Path("../tests/probes/cpap-probe-set.yml").read_text())
gold_probes=[p for p in probes if p.get("gold_evidence")]
qcache=pickle.load(open(".token_economy_r19_qcache.pkl","rb"))
def qkey(q): return hashlib.md5(q.encode()).hexdigest()
GEN_K=64
# retrieve BOTH conventions: exact-k (query at exactly k) and generous (query top_k=64, truncate to k)
seeds_exact={}; seeds_gen={}
with Foundry(settings) as f:
    for p in gold_probes:
        v=qcache[qkey(p["question"])]
        seeds_gen[p["id"]]=[x["id"] for x in vector_query(f.driver,v,VEC_INDEX,top_k=GEN_K) if x["id"] in node]
        e8=[x["id"] for x in vector_query(f.driver,v,VEC_INDEX,top_k=8) if x["id"] in node]
        e16=[x["id"] for x in vector_query(f.driver,v,VEC_INDEX,top_k=16) if x["id"] in node]
        seeds_exact[p["id"]]=(e8,e16)
def mean_recall(getids, scorer, render):
    per=[]
    for p in gold_probes:
        ids=getids(p["id"]); golds=p["gold_evidence"]
        if scorer=="router": r=sum(router_present(g,ids) for g in golds)/len(golds)
        else:
            ctx=(lean_render(ids) if render=="lean" else _norm(" ".join(units_of(ids))))
            r=sum(fuzzy_present(g,ctx) for g in golds)/len(golds)
        per.append(r)
    return float(np.mean(per))
rows=[]
# EXACT-k vs GENEROUS-trunc, each under fuzzy(rich units), fuzzy(lean render), router
for conv,g8,g16 in [("exact", lambda pid: seeds_exact[pid][0], lambda pid: seeds_exact[pid][1]),
                     ("generous", lambda pid: seeds_gen[pid][:8], lambda pid: seeds_gen[pid][:16])]:
    for scorer,render in [("fuzzy","lean"),("fuzzy","rich"),("router","rich")]:
        m8=mean_recall(g8,scorer,render); m16=mean_recall(g16,scorer,render)
        rows.append(dict(convention=conv,scorer=scorer,render=render,k8=m8,k16=m16,lever=m16-m8))
rprint("[bold cyan]H195(a) convention audit - arms 1-2 (=H53 vec8/vec16) on neo4j2[/bold cyan]")
rprint("[dim]OLD cached (exact-k, fuzzy/lean): arm1 k8=0.6667  arm2 k16=0.8542  lever=+0.1875 (+28.1%)[/dim]")
for r in rows:
    rprint(f"  {r['convention']:8s} {r['scorer']:6s}/{r['render']:4s}  k8={r['k8']:.4f}  k16={r['k16']:.4f}  lever={r['lever']:+.4f} ({r['lever']/r['k8']:+.1%})")
# convention isolation: exact vs generous, holding scorer/render fixed
def find(conv,sc,rn): return next(r for r in rows if r["convention"]==conv and r["scorer"]==sc and r["render"]==rn)
conv_shift={}
for sc,rn in [("fuzzy","lean"),("fuzzy","rich"),("router","rich")]:
    e=find("exact",sc,rn); g=find("generous",sc,rn)
    conv_shift[f"{sc}/{rn}"]={"k8_shift":g["k8"]-e["k8"],"k16_shift":g["k16"]-e["k16"]}
rprint(f"[cyan]convention shift (generous - exact), same scorer/render:[/cyan] {json.dumps({k:{kk:round(vv,4) for kk,vv in v.items()} for k,v in conv_shift.items()})}")
# verdict: does the top_k=16 promotion (arm2>arm1) survive? any lever sign flip?
lever_signs=[r["lever"]>0 for r in rows]
verdict_flip = not all(lever_signs)
max_conv_shift=max(abs(v[kk]) for v in conv_shift.values() for kk in v)
rprint(f"[bold]{'VERDICT FLIP' if verdict_flip else 'NO VERDICT FLIP'}[/bold] - top_k=16 promotion (arm2>arm1) "
       f"holds in {sum(lever_signs)}/{len(rows)} scorer/convention cells; max convention magnitude shift = {max_conv_shift*100:.1f}pt")
rprint("[yellow]Arm 3 (SOTA rebuild, default neo4j): graph wiped for H158 (4 nodes); cache has recall+n_gold only, no seeds -> NOT RE-SCORABLE[/yellow]")
h195a=dict(round="R19-H195a",utc=STAMP,graph="neo4j2",probe_set="cpap-probe-set (24 probes, 33 golds)",
    cached_old=dict(arm1_k8=0.6667,arm2_k16=0.8542,lever=0.1875,relative="+28.1%",convention="exact-k",scorer="fuzzy/lean"),
    rescore_rows=rows, convention_shift=conv_shift, verdict_flip=verdict_flip, max_convention_shift_pts=max_conv_shift*100,
    arm3_status="NOT RE-SCORABLE (graph wiped for H158; cache lacks seed lists)")
outp=Path(f"../reports/convention-audit-h195a-{STAMP}.json"); outp.write_text(json.dumps(h195a,indent=2))
rprint(f"[dim]saved {outp}[/dim]")


H195(a) convention audit - arms 1-2 (=H53 vec8/vec16) on neo4j2

OLD cached (exact-k, fuzzy/lean): arm1 k8=0.6667  arm2 k16=0.8542  lever=+0.1875 (+28.1%)

exact    fuzzy /lean  k8=0.6667  k16=0.8542  lever=+0.1875 (+28.1%)

exact    fuzzy /rich  k8=0.8125  k16=0.9583  lever=+0.1458 (+17.9%)

exact    router/rich  k8=0.7917  k16=0.9375  lever=+0.1458 (+18.4%)

...

generous router/rich  k8=0.7917  k16=0.9375  lever=+0.1458 (+18.4%)

convention shift (generous - exact), same scorer/render: {"fuzzy/lean": {"k8_shift": 0.0, "k16_shift": 0.0}, 
"fuzzy/rich": {"k8_shift": 0.0, "k16_shift": 0.0}, "router/rich": {"k8_shift": 0.0, "k16_shift": 0.0}}

NO VERDICT FLIP - top_k=16 promotion (arm2>arm1) holds in 6/6 scorer/convention cells; max convention magnitude 
shift = 0.0pt

Arm 3 (SOTA rebuild, default neo4j): graph wiped for H158 (4 nodes); cache has recall+n_gold only, no seeds -> NOT 
RE-SCORABLE

saved ../reports/convention-audit-h195a-20260707T152107Z.json

## Clause (b) - difficulty-engineered wide benchmark v2

In [4]:
# H195(b) - difficulty-engineered wide benchmark v2 on the H188 derivation harness.
# Tiers: DIFFICULTY (catalogue_code + spec_* + doc_spec_proximity + doc_consolidation_spec), COVERAGE (feature names).
# Document-grounded: catalogue/spec golds reused from the H188 union (probes-wide-h188 + h188b, verbatim-verified);
# consolidation golds derived here from products present across >=2 graph-indexed doc TEXTS, value corroborated in >=2 docs.
FEATURE_RULES={"feature_statement","doc_feature_branded","doc_feature_generic"}
def qn(q): return " ".join(q.lower().split())
union={}
for fp in ["../data/processed/probes-wide-h188.json","../data/processed/probes-wide-h188b.json"]:
    for p in json.load(open(fp))["probes"]:
        k=qn(p["question"])
        if k not in union: union[k]=p
# --- consolidation derivation (deterministic, document-grounded, cross-doc corroborated) ---
def is_shortnum(g):
    gg=_norm(g); return bool(re.fullmatch(r"[\d.,]+",gg)) or (len(re.sub(r"[^\d]","",gg))<=2 and not re.search(r"[a-z]",gg))
def noisy(v): return bool(re.search(r"\d{5,}", v.replace(" ","")))
dd=GraphDatabase.driver(os.environ["NEO4J_URI"], auth=("neo4j","kgfoundry"))
with dd.session() as s:
    prod_names=[r["n"] for r in s.run("MATCH (e:Entity) WHERE any(l IN labels(e) WHERE l IN ['CPAPDevice','ProductModel']) AND e.name IS NOT NULL RETURN DISTINCT e.name AS n").data()]
    graph_docs=set(r["nm"] for r in s.run("MATCH (dd:KGFDocument) RETURN dd.name AS nm").data())
dd.close()
PC=Path("../tmp/parser-round-cache")
_texts={pr:json.loads((PC/f"text_{pr}.json").read_text()) for pr in ["docling","pymupdf4llm","pdfplumber","pypdf"]}
DOCS=[dn for dn in _texts["docling"].keys() if dn in graph_docs]
docnorm={dn:_norm(" ".join(_texts[pr].get(dn,"") for pr in _texts)) for dn in DOCS}
rawd={dn:"\n".join(_texts[pr].get(dn,"") for pr in ["pdfplumber","pymupdf4llm","docling"]) for dn in DOCS}
multidoc=[(nm,sorted(set(dn for dn in DOCS if _norm(nm) in docnorm[dn]))) for nm in prod_names if len(nm)>=5]
multidoc=[(nm,ds) for nm,ds in multidoc if len(ds)>=2]
ATTR=[("weight","weight",r"\b\d[\d.,]*\s?(?:kg|lbs?|kilograms?|grams?)\b"),
 ("sound","sound level",r"\b\d[\d.,]*\s?dB\s?\(?A?\)?\b"),("noise","noise level",r"\b\d[\d.,]*\s?dB\s?\(?A?\)?\b"),
 ("pressure","operating pressure range",r"\b\d[\d.,]*\s?[-–]\s?\d[\d.,]*\s?(?:cm\s?H\s?2?\s?O|hpa)\b"),
 ("ramp","ramp time",r"\b\d[\d.,]*\s?(?:min(?:ute)?s?)\b"),
 ("humidif","humidifier water capacity",r"\b\d[\d.,]*\s?(?:ml|millilitres?)\b"),
 ("warrant","warranty period",r"\b\d\s?(?:years?|yr)\b"),
 ("dimension","dimensions",r"\b\d[\d.,]*\s?[x×]\s?\d[\d.,]*\s?[x×]\s?\d[\d.,]*\s?(?:mm|cm)?\b"),
 ("power","power supply",r"\b\d[\d.,]*\s?(?:w|watts?)\b")]
def _near(low,starts,i,w): return any(abs(i-st)<=w for st in starts)
consol=[]; sseen=set()
for nm,ds in multidoc:
    nl=_norm(nm)
    for akw,lab,vre in ATTR:
        if (nm,lab) in sseen: continue
        cand=Counter(); cand_doc={}
        for dn in ds:
            txt=rawd[dn]; low=txt.lower(); starts=[m.start() for m in re.finditer(re.escape(nl),low)]
            if not starts: continue
            found=None
            for am in re.finditer(re.escape(akw),low):
                if not _near(low,starts,am.start(),140): continue
                vm=re.search(vre, txt[max(0,am.start()-15):am.start()+130], re.I)
                if not vm: continue
                val=re.sub(r"\s+"," ",vm.group(0)).strip()
                if len(val)<2 or not re.search(r"\d",val) or is_shortnum(val) or noisy(val): continue
                found=val; break
            if found:
                nd=sum(1 for xdn in ds if fuzzy_present(found, docnorm[xdn]))
                if nd>cand[found]: cand[found]=nd; cand_doc[found]=dn
        best=[(v,c) for v,c in cand.items() if c>=2]
        if best:
            v,c=max(best,key=lambda x:x[1]); sseen.add((nm,lab))
            q=f"What is the {lab} of the {nm}?"
            if qn(q) in union: continue
            consol.append(dict(question=q, gold_evidence=[v], source_document=cand_doc[v], derivation_rule="doc_consolidation_spec",
                               product=nm, attribute=lab, tier="difficulty", corroborating_docs=c, n_docs=len(ds)))
rprint(f"[cyan]consolidation golds (cross-doc corroborated):[/cyan] {len(consol)}")
# --- assemble v2 ---
v2=[]
for p in union.values():
    tier="coverage" if p.get("derivation_rule") in FEATURE_RULES else "difficulty"
    v2.append(dict(question=p["question"], gold_evidence=p["gold_evidence"], source_document=p.get("source_document"),
                   derivation_rule=p.get("derivation_rule"), product=p.get("product"), attribute=p.get("attribute"), tier=tier))
v2.extend(consol)
for i,g in enumerate(v2): g["id"]=f"V{i+1:03d}"
# --- embed missing (consolidation) questions via Bedrock Titan; reuse wide qcache ---
wqc=pickle.load(open(".wide_probes_h188_qcache.pkl","rb"))
def wqk(q): return hashlib.md5(q.encode()).hexdigest()
missing=[g["question"] for g in v2 if wqk(g["question"]) not in wqc]
if missing:
    embs=generate_embeddings([KEnt.create(q[:80],types=["Query"],description=q) for q in missing], settings.embeddings)
    for q,e in zip(missing,embs): wqc[wqk(q)]=e.embedding
# --- measure router recall@8/@16 per tier (generous top_k=64 truncation) ---
recids={}
with Foundry(settings) as f:
    for g in v2:
        v=wqc[wqk(g["question"])]
        recids[g["id"]]=[x["id"] for x in vector_query(f.driver,v,VEC_INDEX,top_k=64) if x["id"] in node]
def tier_recall(sel,k): return float(np.mean([router_present(g["gold_evidence"][0], recids[g["id"]][:k]) for g in sel])) if sel else None
diff=[g for g in v2 if g["tier"]=="difficulty"]; cons=[g for g in v2 if g["derivation_rule"]=="doc_consolidation_spec"]
core=[g for g in diff if g["derivation_rule"]!="doc_consolidation_spec"]; cov=[g for g in v2 if g["tier"]=="coverage"]
rprint(f"[bold cyan]H195(b) v2 wide benchmark[/bold cyan]  total={len(v2)} golds  (difficulty {len(diff)} | coverage {len(cov)})")
summ={}
for name,sel in [("difficulty(cat+spec+consol)",diff),("  core(catalogue+spec)",core),("  consolidation",cons),("coverage(features)",cov)]:
    r8=tier_recall(sel,8); r16=tier_recall(sel,16); summ[name.strip()]=dict(n=len(sel),r8=r8,r16=r16,lever=(r16-r8) if r8 is not None else None)
    lv=f"{100*(r16-r8):+.1f}pt" if r8 is not None else "-"
    rprint(f"  {name:28s} n={len(sel):3d}  r@8={r8:.3f}  r@16={r16:.3f}  lever={lv}")
byrule={}
for r in sorted(set(g["derivation_rule"] for g in diff)):
    sel=[g for g in diff if g["derivation_rule"]==r]
    byrule[r]=dict(n=len(sel),r8=tier_recall(sel,8),r16=tier_recall(sel,16))
    rprint(f"      {r:24s} n={len(sel):3d}  r@8={byrule[r]['r8']:.3f}  r@16={byrule[r]['r16']:.3f}")
# --- save benchmark + report ---
meta=dict(n=len(v2), date=STAMP, graph="neo4j2", scorer="H194 router (comparator + word-overlap)",
    convention="HNSW top_k=64 truncated to k", tiers={t:sum(1 for g in v2 if g["tier"]==t) for t in ("difficulty","coverage")},
    source="H188-harness union (catalogue+spec, verbatim-verified) + cross-doc-corroborated consolidation golds",
    difficulty_headline=dict(core_r8=summ["core(catalogue+spec)"]["r8"], core_lever=summ["core(catalogue+spec)"]["lever"],
                             with_consolidation_r8=summ["difficulty(cat+spec+consol)"]["r8"], with_consolidation_lever=summ["difficulty(cat+spec+consol)"]["lever"]))
Path("../data/processed/probes-wide-v2-h195.json").write_text(json.dumps(dict(meta=meta, probes=v2), indent=2))
Path(f"../reports/wide-probes-v2-h195-{STAMP}.json").write_text(json.dumps(dict(meta=meta, tier_summary=summ, by_rule=byrule), indent=2))
rprint(f"[dim]saved data/processed/probes-wide-v2-h195.json ({len(v2)} golds) + reports/wide-probes-v2-h195-{STAMP}.json[/dim]")


consolidation golds (cross-doc corroborated): 7

2026-07-07 17:21:15.563 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 7/7
2026-07-07 17:21:15.565 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 7/7 entities via bedrock (0 cache hits)


H195(b) v2 wide benchmark  total=219 golds  (difficulty 117 | coverage 102)

difficulty(cat+spec+consol)  n=117  r@8=0.641  r@16=0.718  lever=+7.7pt

core(catalogue+spec)       n=110  r@8=0.618  r@16=0.700  lever=+8.2pt

...

doc_consolidation_spec   n=  7  r@8=1.000  r@16=1.000

doc_spec_proximity       n= 16  r@8=0.750  r@16=0.875

spec_sentence            n= 12  r@8=0.667  r@16=0.750

spec_table_cell          n= 20  r@8=0.550  r@16=0.700

saved data/processed/probes-wide-v2-h195.json (219 golds) + reports/wide-probes-v2-h195-20260707T152107Z.json